In [0]:
# Questo notebook raccoglie i dati meteo live da Open-Meteo.
#
# Viene eseguito periodicamente, normalmente ogni 15 minuti,
# e salva ogni acquisizione nella tabella Bronze:
# progetto_meteo.bronze.meteo_streaming
#
# La tabella streaming contiene valori istantanei.
# Per questo motivo la temperatura corrente viene salvata come Temp_Istantanea.
#
# Il giorno successivo, il notebook Patcher leggerà questi dati live,
# farà la media per città, data e ora, e trasformerà Temp_Istantanea in Temp_Oraria.
#
# Nota importante:
# all'inizio controllo che la tabella silver.dati_aggiornati sia già pronta.
# Se la Silver meteo non è stata inizializzata dal Clonatore,
# il notebook si ferma e non salva dati live incompleti.


# Controllo iniziale.
# Evito di raccogliere dati live se la Silver meteo non è ancora pronta.
catalogo = "progetto_meteo"
tabella_dati_aggiornati = f"{catalogo}.silver.dati_aggiornati"

righe_silver = spark.table(tabella_dati_aggiornati).limit(1).count()

if righe_silver == 0:
    dbutils.notebook.exit("Silver non ancora pronta: salto questa raccolta live.")


import requests

from datetime import datetime
from zoneinfo import ZoneInfo

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    LongType,
    DoubleType,
    TimestampType
)


# Imposto i riferimenti principali del progetto.
catalogo = "progetto_meteo"
schema_metadati = "metadati"
schema_bronze = "bronze"

tabella_citta = f"{catalogo}.{schema_metadati}.citta_monitorate"
tabella_streaming = f"{catalogo}.{schema_bronze}.meteo_streaming"


# Definisco lo schema della tabella meteo_streaming.
# Uso uno schema esplicito per mantenere tipi e colonne sempre coerenti.
schema_streaming = StructType([
    StructField("Citta", StringType(), True),
    StructField("Regione", StringType(), True),
    StructField("Area", StringType(), True),
    StructField("Data", DateType(), True),
    StructField("Ora", LongType(), True),
    StructField("Temp_Max", DoubleType(), True),
    StructField("Temp_Min", DoubleType(), True),
    StructField("Temp_Istantanea", DoubleType(), True),
    StructField("Umidita", DoubleType(), True),
    StructField("Velocita_Vento", DoubleType(), True),
    StructField("Precipitazioni", DoubleType(), True),
    StructField("Timestamp", TimestampType(), True)
])


# Converto i numeri ricevuti dall'API in float.
# Se il valore manca, lascio None.
def numero(valore):
    if valore is None:
        return None
    return float(valore)


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Leggo le città da monitorare.
# Ogni città contiene coordinate geografiche e macroarea finale.
df_citta = (
    spark.table(tabella_citta)
    .select(
        "citta",
        "regione",
        "area",
        "latitudine",
        "longitudine"
    )
    .orderBy("citta")
)

lista_citta = [riga.asDict() for riga in df_citta.collect()]


# Preparo latitudini e longitudini per fare una sola chiamata Open-Meteo.
# Open-Meteo permette di interrogare più coordinate nella stessa richiesta.
latitudini = ",".join([str(citta["latitudine"]) for citta in lista_citta])
longitudini = ",".join([str(citta["longitudine"]) for citta in lista_citta])


# Imposto endpoint e parametri della Forecast API.
# I nomi dei parametri restano in inglese perché sono quelli richiesti da Open-Meteo.
endpoint_forecast = "https://api.open-meteo.com/v1/forecast"

parametri = {
    "latitude": latitudini,
    "longitude": longitudini,
    "current": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
    "daily": "temperature_2m_max,temperature_2m_min",
    "timezone": "Europe/Rome"
}


# Eseguo la chiamata API verso Open-Meteo.
risposta = requests.get(endpoint_forecast, params=parametri, timeout=120)
risposta.raise_for_status()

dati_json = risposta.json()
lista_risposte_api = dati_json if isinstance(dati_json, list) else [dati_json]


# Creo il timestamp dell'acquisizione.
# Tolgo microsecondi e timezone perché la tabella usa Timestamp standard.
timestamp = datetime.now(ZoneInfo("Europe/Rome")).replace(microsecond=0, tzinfo=None)
data = timestamp.date()
ora = timestamp.hour


# Trasformo i dati live ricevuti in righe per la tabella meteo_streaming.
# Qui current.temperature_2m diventa Temp_Istantanea.
righe_streaming = []

for citta, dati_citta in zip(lista_citta, lista_risposte_api):

    valori_attuali = dati_citta["current"]
    valori_giornalieri = dati_citta["daily"]


    # Creo una mappa giornaliera per recuperare Temp_Max e Temp_Min.
    mappa_giorni = {}

    for giorno, temp_max, temp_min in zip(
        valori_giornalieri["time"],
        valori_giornalieri["temperature_2m_max"],
        valori_giornalieri["temperature_2m_min"]
    ):
        mappa_giorni[giorno] = {
            "Temp_Max": temp_max,
            "Temp_Min": temp_min
        }


    # Recupero Temp_Max e Temp_Min del giorno corrente.
    dati_giorno = mappa_giorni.get(str(data), {
        "Temp_Max": None,
        "Temp_Min": None
    })


    # Creo la riga live della città corrente.
    righe_streaming.append({
        "Citta": citta["citta"],
        "Regione": citta["regione"],
        "Area": citta["area"],
        "Data": data,
        "Ora": ora,
        "Temp_Max": numero(dati_giorno["Temp_Max"]),
        "Temp_Min": numero(dati_giorno["Temp_Min"]),
        "Temp_Istantanea": numero(valori_attuali.get("temperature_2m")),
        "Umidita": numero(valori_attuali.get("relative_humidity_2m")),
        "Velocita_Vento": numero(valori_attuali.get("wind_speed_10m")),
        "Precipitazioni": numero(valori_attuali.get("precipitation")),
        "Timestamp": timestamp
    })


# Creo il DataFrame finale usando lo schema esplicito.
df_streaming = spark.createDataFrame(righe_streaming, schema=schema_streaming)


# Appendo i nuovi dati live nella tabella Delta Bronze.
# Uso append perché ogni esecuzione del notebook aggiunge una nuova acquisizione.
df_streaming.write.mode("append").format("delta").saveAsTable(
    tabella_streaming
)


# Controllo finale.
# Mostro le righe appena acquisite.
display(df_streaming)


# Stampo un riepilogo finale della raccolta live.
print(f"Aggiornamento forecast completato. Tabella aggiornata: {tabella_streaming}")
print(f"Righe aggiunte: {df_streaming.count()}")

# COMMAND ----------